# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [21]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-2.5-flash'
openai = OpenAI(api_key=api_key, 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [3]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [4]:
ed = Website("https://edwarddonner.com")
ed.links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 'https://edwarddonner.com/2025/05/18/2025-ai-executive-briefing/',
 'https://edwarddonner.com/2025/04/21/the-complete-agentic-ai-engineering-course/',
 'https://edwarddonner.com/2025/04/21/the-

## First step: Have Gemini-2.5-flash figure out which links are relevant

### Use a call to gemini-2.5-flash to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}



In [7]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [8]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://edwarddonner.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://edwarddo

In [22]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [10]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/baidu/ERNIE-4.5-21B-A3B-Thinking',
 '/tencent/HunyuanImage-2.1',
 '/Qwen/Qwen3-Next-80B-A3B-Instruct',
 '/google/embeddinggemma-300m',
 '/tencent/SRPO',
 '/models',
 '/spaces/enzostvs/deepsite',
 '/spaces/zerogpu-aoti/wan2-2-fp8da-aoti-faster',
 '/spaces/multimodalart/wan-2-2-first-last-frame',
 '/spaces/Qwen/Qwen3-ASR-Demo',
 '/spaces/tencent/HunyuanImage-2.1',
 '/spaces',
 '/datasets/HuggingFaceFW/finepdfs',
 '/datasets/HuggingFaceM4/FineVision',
 '/datasets/fka/awesome-chatgpt-prompts',
 '/datasets/JDhruv14/Bhagavad-Gita_Dataset',
 '/datasets/LucasFang/FLUX-Reason-6M',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google',
 '/Intel',
 '/microsoft',
 '/grammarly',
 '/Wri

In [23]:
get_links("https://huggingface.co")

{'links': [{'type': 'enterprise solutions',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'models platform', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets platform', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces platform', 'url': 'https://huggingface.co/spaces'},
  {'type': 'brand information', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'learn/education', 'url': 'https://huggingface.co/learn'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'github profile', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin profile',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'discord community', 'url': 'ht

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [24]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [25]:
print(get_all_details("https://huggingface.co"))

Found links: {'links': [{'type': 'documentation', 'url': 'https://huggingface.co/docs'}, {'type': 'enterprise solutions', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing', 'url': 'https://huggingface.co/pricing'}, {'type': 'company updates', 'url': 'https://huggingface.co/changelog'}, {'type': 'brand information', 'url': 'https://huggingface.co/brand'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'learning resources', 'url': 'https://huggingface.co/learn'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'community forum', 'url': 'https://discuss.huggingface.co'}, {'type': 'github profile', 'url': 'https://github.com/huggingface'}, {'type': 'twitter profile', 'url': 'https://twitter.com/huggingface'}, {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'discord community', 'url': 'https://huggingface.co/join/discord'}]}
Landing page:
Webpage Title:
Hugging Face – The AI c

In [26]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."


In [27]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [28]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'company website', 'url': 'https://huggingface.co/'}, {'type': 'product page', 'url': 'https://huggingface.co/models'}, {'type': 'product page', 'url': 'https://huggingface.co/datasets'}, {'type': 'product page', 'url': 'https://huggingface.co/spaces'}, {'type': 'documentation', 'url': 'https://huggingface.co/docs'}, {'type': 'enterprise solutions', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing', 'url': 'https://huggingface.co/pricing'}, {'type': 'product offering', 'url': 'https://endpoints.huggingface.co'}, {'type': 'product feature', 'url': 'https://huggingface.co/chat'}, {'type': 'brand information', 'url': 'https://huggingface.co/brand'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'learning resources', 'url': 'https://huggingface.co/learn'}, {'type': 'company blog', 'url': 'https://huggingface.co/blog'}]}


'You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\nWebpage Contents:\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nbaidu/ERNIE-4.5-21B-A3B-Thinking\nUpdated\n3 days ago\n•\n100k\n•\n660\ntencent/HunyuanImage-2.1\nUpdated\n2 days ago\n•\n949\n•\n574\nQwen/Qwen3-Next-80B-A3B-Instruct\nUpdated\n3 days ago\n•\n142k\n•\n466\ngoogle/embeddinggemma-300m\nUpdated\n4 days ago\n•\n153k\n•\n768\ntencent/SRPO\nUpdated\n1 day ago\n•\n2.53k\n•\n353\nBrowse 1M+ models\nSpaces\nRunning\n13.3k\n13.3k\nDeepSite v2\n🐳\nG

In [29]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [30]:
create_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'product category', 'url': 'https://huggingface.co/models'}, {'type': 'product category', 'url': 'https://huggingface.co/datasets'}, {'type': 'product category', 'url': 'https://huggingface.co/spaces'}, {'type': 'documentation', 'url': 'https://huggingface.co/docs'}, {'type': 'enterprise solutions', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing', 'url': 'https://huggingface.co/pricing'}, {'type': 'news & updates', 'url': 'https://huggingface.co/changelog'}, {'type': 'product offering', 'url': 'https://endpoints.huggingface.co'}, {'type': 'product offering', 'url': 'https://huggingface.co/chat'}, {'type': 'about us', 'url': 'https://huggingface.co/huggingface'}, {'type': 'brand information', 'url': 'https://huggingface.co/brand'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'learning resources', 'url': 'https://huggingface.co/learn'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}]}


## Hugging Face: The AI Community Building the Future

**Hugging Face** is the leading platform where the machine learning community collaborates, builds, and deploys state-of-the-art AI. We are dedicated to accelerating the advancement and accessibility of machine learning through open science, shared resources, and powerful tooling.

---

### What We Offer

**The Hugging Face Hub:** Your central platform for everything ML.
*   **1M+ Models:** Discover and share cutting-edge AI models across all modalities – text, image, video, audio, and even 3D.
*   **250k+ Datasets:** Access a vast collection of datasets for training and evaluating your machine learning projects.
*   **400k+ Spaces (AI Apps):** Host and explore interactive demos and applications, showcasing the power of AI.

**Open Source Innovation:** We are building the foundational ML tooling with the community. Our robust open-source libraries include:
*   **Transformers:** State-of-the-art AI models for PyTorch.
*   **Diffusers:** Pioneering diffusion models for generative AI.
*   **Datasets:** Tools to easily access and share datasets.
*   **PEFT, Accelerate, TRL, Tokenizers, Safetensors,** and many more, empowering developers to move faster and build better.

**Accelerate Your ML (Paid Solutions):**
*   **Compute:** Deploy your models on optimized Inference Endpoints or elevate your Spaces applications with GPU power, starting at just $0.60/hour.
*   **Team & Enterprise:** Equip your organization with the most advanced platform to build AI. Benefit from enterprise-grade security, access controls, dedicated support, Single Sign-On (SSO), audit logs, and private dataset viewing, starting at $20/user/month.

---

### Who Uses Hugging Face?

Hugging Face is trusted by over **50,000 organizations** worldwide, from individual developers and startups to leading global enterprises:

*   **Tech Giants:** Google, Microsoft, Meta, Amazon, Intel
*   **Innovative Companies:** Grammarly, Writer
*   **Research & Non-Profits:** Ai2

We empower a diverse range of users to create, discover, and collaborate on ML better, building their portfolios and sharing their work with the world.

---

### Our Community & Culture

At Hugging Face, we believe in the power of **community and collaboration**. Our culture is rooted in:
*   **Openness:** Fostering an environment where knowledge and resources are shared freely.
*   **Innovation:** Constantly pushing the boundaries of what's possible in AI.
*   **Empowerment:** Providing tools and platforms that enable everyone to contribute to the future of machine learning.

We are a community-first company, dedicated to supporting researchers, developers, and organizations in their AI journey.

---

### Join Our Team

Are you passionate about artificial intelligence and open source? Do you want to contribute to a platform that is shaping the future of machine learning?

**Explore career opportunities** at Hugging Face and help us build the next generation of AI tools and community.

---

**Learn more and get started today at [huggingface.co](https://huggingface.co)**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [31]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [32]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'home page', 'url': 'https://huggingface.co/'}, {'type': 'models catalog', 'url': 'https://huggingface.co/models'}, {'type': 'datasets catalog', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces (apps/demos)', 'url': 'https://huggingface.co/spaces'}, {'type': 'enterprise solutions', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing', 'url': 'https://huggingface.co/pricing'}, {'type': 'brand information', 'url': 'https://huggingface.co/brand'}, {'type': 'learning resources', 'url': 'https://huggingface.co/learn'}, {'type': 'documentation', 'url': 'https://huggingface.co/docs'}, {'type': 'company blog', 'url': 'https://huggingface.co/blog'}, {'type': 'chat product', 'url': 'https://huggingface.co/chat'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'community forum', 'url': 'https://discuss.huggingface.co'}, {'type': 'endpoints product', 'url': 'https://endpoints.huggingface.co'}, {'type': 'gi

# Hugging Face: The AI Community Building the Future

Hugging Face is the leading platform where the machine learning (ML) community collaborates on models, datasets, and applications, truly **building the future of AI together**. We are the home of open-source ML, fostering innovation across all modalities – from text and images to video, audio, and 3D.

---

## The Heart of ML Collaboration: The Hugging Face Hub

Our expansive Hub is the central hub where the world's ML experts converge to discover, create, and share.

*   **Models:** Browse over **1 million models**, including state-of-the-art developments from industry leaders like Google, Tencent, and Baidu.
*   **Datasets:** Access and share from a vast collection of over **250,000 datasets** to fuel your ML projects.
*   **Spaces:** Deploy and showcase your applications with ease, exploring over **400,000 interactive ML demos**.

This open platform allows you to host, collaborate, and build your ML portfolio effortlessly.

---

## Driving Innovation with Open Source

We are deeply committed to building the foundational ML tooling with and for the community. Our robust open-source libraries are cornerstones of modern AI development:

*   **Transformers:** For state-of-the-art AI models in PyTorch.
*   **Diffusers:** Enabling cutting-edge diffusion models.
*   **Datasets & Tokenizers:** Essential tools for efficient data handling and processing.
*   **Accelerate & PEFT:** Streamlining model training and parameter-efficient fine-tuning for large language models.

By providing these tools, we empower developers to move faster and innovate on the bleeding edge of AI.

---

## Accelerate Your ML with Enterprise-Grade Solutions

For teams and organizations, Hugging Face offers powerful solutions to scale and secure your AI initiatives:

*   **Compute:** Deploy models on optimized **Inference Endpoints** or upgrade your **Spaces applications** to a GPU in a few clicks, starting from just $0.60/hour.
*   **Team & Enterprise:** Gain advanced features like enterprise-grade security, comprehensive access controls, dedicated support, Single Sign-On (SSO), audit logs, resource groups, and private dataset viewing.

### Trusted by the World's Leading Innovators

Over **50,000 organizations** rely on Hugging Face, including industry giants such as:
**Google • Microsoft • Amazon • Meta • Intel • Grammarly • Ai2**

---

## Join the Community, Build the Future

At Hugging Face, we believe in the power of collaboration and an open approach to AI. Our vibrant community is at the heart of everything we do, pushing the boundaries of what's possible in machine learning.

*   **Company Culture:** We champion an **open, collaborative, and innovation-driven environment**, where contributions from everyone are valued. We are a community-first organization, dedicated to building the foundation of ML tooling *with the community*.
*   **Careers:** Passionate about AI and open source? Join our diverse team and contribute to shaping the future of machine learning. Explore exciting career opportunities on our website.

---

**Discover, Collaborate, Innovate with Hugging Face.**

Visit **huggingface.co** to explore models, datasets, and applications, or to learn more about our enterprise offerings and career opportunities.

In [33]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'company homepage', 'url': 'https://huggingface.co'}, {'type': 'company profile page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'enterprise solutions page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'company blog', 'url': 'https://huggingface.co/blog'}, {'type': 'learning resources page', 'url': 'https://huggingface.co/learn'}, {'type': 'documentation page', 'url': 'https://huggingface.co/docs'}, {'type': 'changelog page', 'url': 'https://huggingface.co/changelog'}, {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'twitter profile', 'url': 'https://twitter.com/huggingface'}, {'type': 'github profile', 'url': 'https://github.com/huggingface'}, {'type': 'community forum', 'url': 'https://discuss.huggingface.co'}, {'type': 'status page', 'url': 

# Hugging Face: The AI Community Building the Future

**Hugging Face** is the leading platform where the global machine learning (ML) community collaborates, discovers, and builds the future of artificial intelligence. We empower individuals and organizations to create, share, and accelerate their AI initiatives.

---

### **What We Do**

At our core, we foster open collaboration and provide the essential tools and infrastructure for modern AI development.

*   **The Hugging Face Hub:**
    *   **Models:** Browse and host over 1 million state-of-the-art ML models across all modalities (text, image, video, audio, 3D).
    *   **Datasets:** Access and share over 250,000 datasets for every ML task.
    *   **Spaces:** Run and discover over 400,000 interactive ML applications and demos.
    *   **Open Source Stack:** We are building the foundation of ML tooling with the community, including popular libraries like Transformers, Diffusers, Datasets, PEFT, and many more, with millions of downloads and community contributions.

*   **Accelerate Your ML:**
    *   **Compute:** Deploy on optimized Inference Endpoints and upgrade your Spaces applications to GPUs with ease.
    *   **Team & Enterprise Solutions:** For organizations, we offer enterprise-grade security, advanced access controls, Single Sign-On, audit logs, private datasets, and dedicated support, empowering teams to build AI with confidence.

---

### **Our Community & Customers**

Hugging Face is truly "the AI community building the future." We are home to a vibrant, collaborative ecosystem where researchers, developers, and companies come together to innovate.

*   **50,000+ Organizations** trust Hugging Face, including industry giants like **Google, Meta, Amazon, Microsoft, Intel**, as well as innovative companies like **Grammarly** and **Writer**, and leading research institutions like **Ai2**.
*   We believe in the power of open science and shared resources to democratize AI and move the field forward faster.

---

### **Company Culture**

Our culture is built on **openness, collaboration, and a passion for ML innovation**. We are a community-driven organization committed to building the foundational tooling that empowers everyone to participate in the AI revolution. We value contributions from all, fostering an environment where sharing knowledge and building together is paramount.

---

### **Join Us!**

*   **For Customers:** Whether you're an individual developer, a startup, or an enterprise, Hugging Face provides the platform and tools to accelerate your ML journey. Explore our offerings and pricing today!
*   **For Investors:** Invest in the foundational platform enabling the global AI ecosystem. Hugging Face is at the heart of ML innovation, powering the next generation of AI applications.
*   **For Recruits:** Are you passionate about machine learning and open source? Do you want to work at the cutting edge of AI, building tools that impact millions? Explore our **Jobs** page and join our diverse, talented team committed to building the future of AI.

---

**Hugging Face: Collaborate. Innovate. Build the Future of AI.**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>